# Classification Baseline — Multi-Model Comparison

Predicts **loan outcome** (binary: 1 = approved, 0 = not approved) using LogisticRegression,
RandomForest, and HistGradientBoosting.
Features: `BasicControlFlowFeatures` only (log-based; time-series features excluded).
All run parameters are defined in `configs/classification.yaml`.

In [1]:
import contextlib
import logging
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

from spi_time_series.config import load_config
from spi_time_series.config.loader import build_estimator
from spi_time_series.data.schemas import (
    EvaluationReport,
    ModelArtifact,
    RawData,
)
from spi_time_series.features.extraction import extract_features_builder
from spi_time_series.features.log_based_features import BasicControlFlowFeatures
from spi_time_series.models.trainer import train
from spi_time_series.pipeline import PipelineBuilder
from spi_time_series.pipeline.checkpointing import checkpoint

logging.basicConfig(level=logging.INFO, force=True)

## Config

In [2]:
config = load_config(Path("../configs/classification.yaml"))

MODEL_CONFIGS = {
    name: (build_estimator(m), m.param_grid)
    for name, m in config.models.items()
}

## Target generator

The loan outcome is stored in the `Accepted` column of offer events — it is **not** determined
by the last activity. The label is derived by scanning the **full trace**: if any event has
`Accepted == True` the case is positive. Every prefix of the same case gets the same label.

`_col_idx_store` is populated from `preprocessed.col_idx` after splitting and read by
`loan_outcome` during feature extraction.

In [3]:
_col_idx_store: dict[str, int] = {}


def loan_outcome(trace: np.ndarray, start_idx: int, end_idx: int) -> int:
    accepted_idx = _col_idx_store["Accepted"]
    return 1 if any(v is True for v in trace[:, accepted_idx]) else 0

## Pipeline

In [4]:
pipeline = (
    PipelineBuilder.from_config(config)
    .with_feature_extractor(
        extract_features_builder(
            [
                BasicControlFlowFeatures(
                    one_hot_encode_categorical=config.features.one_hot_encode_categorical,
                )
            ],
            loan_outcome,
        )
    )
    .build()
)

INFO:spi_time_series.data.dataset:Dataset found at G:\spi-time-series\data\raw\BPI Challenge 2017.xes.gz.
INFO:spi_time_series.data.dataset:Reading XES log...
g:\spi-time-series\.venv\Lib\site-packages\pm4py\utils.py:1005: UserWarning: In the current version, the import/export operation uses `r4pm` by default for importing/exporting files faster.
  warnings.warn(
INFO:spi_time_series.data.dataset:Done. Loaded 1202267 events.


## Stage 1 — Load and clean

In [5]:
raw = RawData(event_log=pipeline.dataset.log)

cleaned = checkpoint(
    config.checkpoint_dir / "cleaned_log_clf.pkl",
    lambda: pipeline.preprocessor(raw),
    params=config.checkpoint_params(),
)

INFO:spi_time_series.pipeline.checkpointing:Checkpoint not found — computing: ..\data\checkpoints\cleaned_log_clf__854b8858.pkl
INFO:spi_time_series.preprocessing.preprocess:Filtering for valid end activities: ['W_Validate application', 'W_Call after offers', 'W_Call incomplete files', 'O_Cancelled', 'A_Denied']
INFO:spi_time_series.preprocessing.preprocess:Remaining events after cleaning: 1193604
INFO:spi_time_series.pipeline.checkpointing:Checkpoint saved: ..\data\checkpoints\cleaned_log_clf__854b8858.pkl


## Stage 2 — Split and extract features

Splitting is not checkpointed — it is fast and produces lazy iterators that cannot be serialised.
The column-index store is populated from `preprocessed.col_idx` so that `loan_outcome`
can look up the `Accepted` column index during feature extraction.

In [6]:
preprocessed = pipeline.splitter(cleaned)
_col_idx_store.update(preprocessed.col_idx)

features = checkpoint(
    config.checkpoint_dir / "features_clf.pkl",
    lambda: pipeline.feature_extractor(preprocessed),
    params=config.checkpoint_params(),
)

print(f"Train shape: {features.X_train.shape}")
print(f"Test shape:  {features.X_test.shape}")
print(f"Class balance (train): {features.y_train.value_counts().to_dict()}")

INFO:spi_time_series.preprocessing.preprocess:Splitting cases at cutoff time: 2016-10-18 17:03:13.107600128+00:00
INFO:spi_time_series.preprocessing.preprocess:Split successful. 
Train: 22723 cases, 
Test: 6247 cases.
INFO:spi_time_series.pipeline.checkpointing:Checkpoint not found — computing: ..\data\checkpoints\features_clf__854b8858.pkl
Processing cases: 100%|██████████| 6247/6247 [00:25<00:00, 244.47it/s]
INFO:spi_time_series.pipeline.checkpointing:Checkpoint saved: ..\data\checkpoints\features_clf__854b8858.pkl


Train shape: (822706, 33)
Test shape:  (227073, 33)
Class balance (train): {1: 594871, 0: 227835}


## Stage 3 — Hyperparameter search and fit

CV search runs on a stratified subsample (`config.search.search_sample_size`) for speed;
final models are fit on the full training set.
The artifact is checkpointed so this stage is skipped on re-runs when the config is unchanged.

In [7]:
@contextlib.contextmanager
def _tqdm_joblib(pbar):
    class _CB(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *a, **kw):
            pbar.update(self.batch_size)
            return super().__call__(*a, **kw)

    old = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = _CB
    try:
        yield pbar
    finally:
        joblib.parallel.BatchCompletionCallBack = old


def _fit_artifact() -> ModelArtifact:
    from sklearn.pipeline import Pipeline as _SKPipeline

    search_pre = ColumnTransformer(
        [
            (
                "num",
                _SKPipeline(
                    [
                        ("imp", SimpleImputer(strategy="median")),
                        ("sc", StandardScaler()),
                    ]
                ),
                make_column_selector(dtype_include=np.number),
            )
        ],
        remainder="drop",
    )
    X_num = search_pre.fit_transform(features.X_train)

    sample_size = config.search.search_sample_size
    X_search, _, y_search, _ = train_test_split(
        X_num,
        features.y_train,
        train_size=sample_size,
        stratify=features.y_train,
        random_state=config.search.random_state,
    )
    n_fits = config.search.n_iter * config.search.cv_folds
    print(f"CV search on {len(y_search):,} samples ({n_fits} fits total)")

    best_estimators = {}
    for name, (base_est, hp_grid) in MODEL_CONFIGS.items():
        search = RandomizedSearchCV(
            base_est,
            param_distributions=hp_grid,
            n_iter=config.search.n_iter,
            cv=config.search.cv_folds,
            scoring="roc_auc",
            refit=False,
            n_jobs=-1,
            random_state=config.search.random_state,
        )
        with _tqdm_joblib(tqdm(desc=name, total=n_fits)):
            search.fit(X_search, y_search)
        print(
            f"{name}: best={search.best_params_}  CV AUC={search.best_score_:.4f}"
        )
        best_estimators[name] = clone(base_est).set_params(
            **search.best_params_
        )

    return train(features, best_estimators)


artifact = checkpoint(
    config.checkpoint_dir / "artifact_clf.pkl",
    _fit_artifact,
    params=config.checkpoint_params(),
)

INFO:spi_time_series.pipeline.checkpointing:Checkpoint not found — computing: ..\data\checkpoints\artifact_clf__854b8858.pkl


CV search on 50,000 samples (30 fits total)


logistic_regression: 100%|██████████| 30/30 [01:26<00:00,  3.99s/it]

logistic_regression: best={'C': 1000.0}  CV AUC=0.5714


random_forest: best={'n_estimators': 150, 'min_samples_split': 10, 'max_features': 'sqrt', 'max_depth': 10}  CV AUC=0.5992













INFO:spi_time_series.models.trainer:Training model: logistic_regression


hist_gradient_boosting: best={'max_iter': 100, 'max_depth': None, 'learning_rate': 0.05}  CV AUC=0.5995



INFO:spi_time_series.models.trainer:Model logistic_regression trained.
INFO:spi_time_series.models.trainer:Training model: random_forest
INFO:spi_time_series.models.trainer:Model random_forest trained.
INFO:spi_time_series.models.trainer:Training model: hist_gradient_boosting
hist_gradient_boosting: 100%|██████████| 30/30 [03:27<00:00,  6.92s/it]
INFO:spi_time_series.models.trainer:Model hist_gradient_boosting trained.
INFO:spi_time_series.pipeline.checkpointing:Checkpoint saved: ..\data\checkpoints\artifact_clf__854b8858.pkl


## Stage 4 — Evaluate

In [8]:
_PL_COL = "BasicControlFlowFeatures__prefix_length"


def evaluate_classification(
    artifact: ModelArtifact, feature_set
) -> EvaluationReport:
    X_test, y_test = feature_set.X_test, feature_set.y_test
    groups = X_test.groupby(_PL_COL).groups
    all_metrics: dict = {}

    for name, pipe in artifact.models.items():
        y_prob = pd.Series(pipe.predict_proba(X_test)[:, 1], index=X_test.index)
        y_pred = pd.Series(pipe.predict(X_test), index=X_test.index)
        model_metrics: dict = {}
        for pl, idx in groups.items():
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                try:
                    auc = float(roc_auc_score(y_test.loc[idx], y_prob.loc[idx]))
                except ValueError:
                    auc = float("nan")
                model_metrics[int(pl)] = {
                    "auc": auc,
                    "brier": float(
                        brier_score_loss(y_test.loc[idx], y_prob.loc[idx])
                    ),
                    "accuracy": float(
                        accuracy_score(y_test.loc[idx], y_pred.loc[idx])
                    ),
                }
        all_metrics[name] = model_metrics

    prefix_lengths = sorted(int(pl) for pl in groups)
    return EvaluationReport(
        metrics=all_metrics,
        model_names=list(artifact.models),
        prefix_lengths=prefix_lengths,
    )


report = evaluate_classification(artifact, features)

## Results

AUC, Brier Score, and Accuracy per model and prefix length.

In [9]:
pd.DataFrame(
    [
        {
            "model": model,
            "prefix_length": pl,
            "AUC": round(report.metrics[model][pl]["auc"], 4),
            "Brier Score": round(report.metrics[model][pl]["brier"], 4),
            "Accuracy": round(report.metrics[model][pl]["accuracy"], 4),
        }
        for model in sorted(report.model_names)
        for pl in report.prefix_lengths
    ]
)

,model,prefix_length,AUC,Brier Score,Accuracy
0,hist_gradient_boosting,3,0.5592,0.1902,0.7412
1,hist_gradient_boosting,4,0.6235,0.1757,0.7680
2,hist_gradient_boosting,5,0.6299,0.1748,0.7677
3,hist_gradient_boosting,6,0.6312,0.1752,0.7672
4,hist_gradient_boosting,7,0.6287,0.1757,0.7671
5,hist_gradient_boosting,8,0.6236,0.1763,0.7669
6,hist_gradient_boosting,9,0.6236,0.1768,0.7656
7,hist_gradient_boosting,10,0.5726,0.1894,0.7432
8,logistic_regression,3,0.4740,0.1931,0.7412
9,logistic_regression,4,0.5792,0.1925,0.7412


### AUC by prefix length (pivot)

Rows = prefix length, columns = model.

In [10]:
pd.DataFrame(
    {
        model: {
            pl: round(report.metrics[model][pl]["auc"], 4)
            for pl in report.prefix_lengths
        }
        for model in sorted(report.model_names)
    }
).rename_axis("prefix_length")

,hist_gradient_boosting,logistic_regression,random_forest
prefix_length,,,
3,0.5592,0.4740,0.5147
4,0.6235,0.5792,0.6239
5,0.6299,0.5865,0.6329
6,0.6312,0.5910,0.6316
7,0.6287,0.5777,0.6237
8,0.6236,0.5740,0.6171
9,0.6236,0.5754,0.6198
10,0.5726,0.5495,0.5691
